<a href="https://colab.research.google.com/github/yoshiagent/IntControl-colab/blob/main/notebooks/Fuzzy_toolbox.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="在 Colab 開啟"/></a>

# 模糊邏輯工具箱：原課程 MATLAB 指令的 Python 版

原課程用 MATLAB 的 Fuzzy Logic Toolbox（`readfis`、`evalfis`、`newfis`、`addvar`、`addmf`、`addrule`、`showrule`、`gensurf`…）示範模糊推論系統（FIS）。
這本筆記本用 numpy 寫出一個功能相同的小型工具箱，讀原講義的 .fis 檔、重現網頁上每一段指令的結果。對照的網頁：
[Fuzzy Logic Toolbox](https://yoshiagent.github.io/IntControl/FZ-FuzzyLogicToolbox.html)、
[以指令操作 FIS](https://yoshiagent.github.io/IntControl/FZ-FisCmdFlash.html)、
[從頭建立 FIS（CreateFIS.m）](https://yoshiagent.github.io/IntControl/FZ-FisScratch.html)、
[刪除歸屬函數、變數與規則](https://yoshiagent.github.io/IntControl/FZ-FISRMFlash.html)、
[Mamdani 型推論](https://yoshiagent.github.io/IntControl/FZ-mamdani.html)、[Sugeno 型推論](https://yoshiagent.github.io/IntControl/FZ-Sugeno.html)。

推論的算法和 MATLAB 相同（Mamdani 的輸出範圍切 101 點做解模糊化，這是 `evalfis` 的預設），所以數字和網頁、和 MATLAB 一致。只用 Colab 內建的 numpy 與 plotly。

In [1]:
import re
import urllib.request
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

SITE = "https://yoshiagent.github.io/IntControl/files/2Controller/fuzzy/"

def fetch(name):
    """從課程網站下載原講義的附檔到目前的資料夾（Colab 需要連網）"""
    with urllib.request.urlopen(SITE + name, timeout=30) as r:
        open(name, "wb").write(r.read())
    return name

for f in ("tipper.fis", "tipper0.fis", "tippersg.fis", "MyTipper.fis"):
    fetch(f)
print(open("tipper.fis", encoding="latin-1").read())

% $Revision: 1.1 $
[System]
Name='tipper'
Type='mamdani'
NumInputs=2
NumOutputs=1
NumRules=3
AndMethod='min'
OrMethod='max'
ImpMethod='min'
AggMethod='max'
DefuzzMethod='centroid'

[Input1]
Name='service'
Range=[0 10]
NumMFs=3
MF1='poor':'gaussmf',[1.5 0]
MF2='good':'gaussmf',[1.5 5]
MF3='excellent':'gaussmf',[1.5 10]

[Input2]
Name='food'
Range=[0 10]
NumMFs=2
MF1='rancid':'trapmf',[0 0 1 3]
MF2='delicious':'trapmf',[7 9 10 10]

[Output1]
Name='tip'
Range=[0 30]
NumMFs=3
MF1='cheap':'trimf',[0 5 10]
MF2='average':'trimf',[10 15 20]
MF3='generous':'trimf',[20 25 30]

[Rules]
1 1, 1 (1) : 2 
2 0, 2 (1) : 1 
3 2, 3 (1) : 2 



## 1. 小型模糊工具箱

FIS 用 Python 的 dict 表示，欄位名稱和 MATLAB 舊版的 FIS 結構相同：`fis["input"][0]["mf"][1]["params"]` 就是 MATLAB 的 `fis.input(1).mf(2).params`（索引從 0 開始）。

### 歸屬函數

和工具箱的定義相同：`trimf [a b c]`、`trapmf [a b c d]`、`gaussmf [σ c]`、`gauss2mf [σ₁ c₁ σ₂ c₂]`、`gbellmf [a b c]`、`sigmf [a c]`、`pimf`、`zmf`、`smf`；Sugeno 的輸出用 `constant [z]` 或 `linear [p₁ … pₙ r]`。

In [2]:
def mf(kind, p, x):
    x = np.asarray(x, float)
    if kind == "trimf":
        a, b, c = p
        l = (x - a) / (b - a) if b > a else (x >= b) * 1.0
        r = (c - x) / (c - b) if c > b else (x <= b) * 1.0
        return np.clip(np.minimum(l, r), 0, 1)
    if kind == "trapmf":
        a, b, c, d = p
        l = (x - a) / (b - a) if b > a else (x >= b) * 1.0
        r = (d - x) / (d - c) if d > c else (x <= c) * 1.0
        return np.clip(np.minimum(np.minimum(l, 1.0), r), 0, 1)
    if kind == "gaussmf":
        s, c = p
        return np.exp(-(x - c) ** 2 / (2 * s * s))
    if kind == "gauss2mf":
        s1, c1, s2, c2 = p
        return np.where(x < c1, np.exp(-(x - c1) ** 2 / (2 * s1 * s1)), 1.0) * np.where(x > c2, np.exp(-(x - c2) ** 2 / (2 * s2 * s2)), 1.0)
    if kind == "gbellmf":
        a, b, c = p[:3]
        return 1 / (1 + np.abs((x - c) / a) ** (2 * b))
    if kind == "sigmf":
        a, c = p
        return 1 / (1 + np.exp(-a * (x - c)))
    if kind in ("zmf", "smf"):
        a, b = p
        m = (a + b) / 2
        y = np.where(x <= a, 0.0, np.where(x <= m, 2 * ((x - a) / (b - a)) ** 2, np.where(x <= b, 1 - 2 * ((x - b) / (b - a)) ** 2, 1.0)))
        return 1 - y if kind == "zmf" else y
    if kind == "pimf":
        a, b, c, d = p
        return mf("smf", [a, b], x) * mf("zmf", [c, d], x)
    raise ValueError(kind)

### 建立與修改：newfis、addvar、addmf、addrule、rmmf、rmvar

規則用和 MATLAB 相同的數字表示：每列是「各輸入的歸屬函數編號…、輸出的歸屬函數編號、權重、連接詞（1 = AND、2 = OR）」，編號 0 表示不用這個變數，負號表示 NOT。

In [3]:
def newfis(name, fis_type="mamdani"):
    return dict(name=name, type=fis_type, andMethod="min" if fis_type == "mamdani" else "prod", orMethod="max",
                impMethod="min", aggMethod="max", defuzzMethod="centroid" if fis_type == "mamdani" else "wtaver",
                input=[], output=[], rule=[])

def addvar(fis, var_type, name, rng):
    fis[var_type].append(dict(name=name, range=list(map(float, rng)), mf=[]))
    for r in fis["rule"]:                                   # 新增輸入時，舊規則對它都不設條件
        if var_type == "input":
            r["antecedent"].append(0)
        else:
            r["consequent"].append(0)
    return fis

def addmf(fis, var_type, var_index, name, kind, params):
    """var_index 和 MATLAB 一樣從 1 開始"""
    fis[var_type][var_index - 1]["mf"].append(dict(name=name, type=kind, params=list(map(float, params))))
    return fis

def addrule(fis, rulelist):
    n_in = len(fis["input"])
    for row in np.atleast_2d(rulelist):
        row = [float(v) for v in row]
        fis["rule"].append(dict(antecedent=[int(v) for v in row[:n_in]], consequent=[int(v) for v in row[n_in:-2]],
                                weight=row[-2], connection=int(row[-1])))
    return fis

def rmmf(fis, var_type, var_index, mf_index):
    """刪除第 var_index 個變數的第 mf_index 個歸屬函數（都從 1 開始）。
    規則是以編號記錄的：用到這個歸屬函數的條件改成 0（不用），比它大的編號減 1，並列出受影響的規則"""
    fis[var_type][var_index - 1]["mf"].pop(mf_index - 1)
    key = "antecedent" if var_type == "input" else "consequent"
    for i, r in enumerate(fis["rule"]):
        v = r[key][var_index - 1]
        if abs(v) == mf_index:
            r[key][var_index - 1] = 0
            print(f"  注意：規則 {i + 1} 用到被刪除的歸屬函數，這個條件已改成 0（不用）")
        elif abs(v) > mf_index:
            r[key][var_index - 1] = int(np.sign(v) * (abs(v) - 1))
    return fis

def rmvar(fis, var_type, var_index):
    """刪除變數，並把每條規則裡對應的那一欄拿掉"""
    fis[var_type].pop(var_index - 1)
    key = "antecedent" if var_type == "input" else "consequent"
    for r in fis["rule"]:
        r[key].pop(var_index - 1)
    return fis

### 讀寫 .fis 檔：readfis、writefis

In [4]:
def readfis(path):
    path = path if path.endswith(".fis") else path + ".fis"     # 和 MATLAB 一樣可以省略副檔名
    fis, sec, cur = newfis(""), None, None
    for line in open(path, encoding="latin-1").read().splitlines():
        line = line.strip()
        if not line or line.startswith("%"):
            continue
        m = re.match(r"\[(\w+?)(\d*)\]", line)
        if m:
            sec = m.group(1)
            if sec in ("Input", "Output"):
                cur = dict(name="", range=[0.0, 1.0], mf=[])
                fis["input" if sec == "Input" else "output"].append(cur)
            continue
        if sec == "Rules":
            a, rest = line.split(",")
            c, w, conn = re.match(r"\s*([-\d.\s]+)\(([\d.]+)\)\s*:\s*(\d)", rest).groups()
            fis["rule"].append(dict(antecedent=[int(float(v)) for v in a.split()], consequent=[int(float(v)) for v in c.split()],
                                    weight=float(w), connection=int(conn)))
            continue
        k, v = (s.strip() for s in line.split("=", 1))
        if sec == "System":
            key = {"Name": "name", "Type": "type", "AndMethod": "andMethod", "OrMethod": "orMethod", "ImpMethod": "impMethod",
                   "AggMethod": "aggMethod", "DefuzzMethod": "defuzzMethod"}.get(k)
            if key:
                fis[key] = v.strip("'")
        elif k == "Name":
            cur["name"] = v.strip("'")
        elif k == "Range":
            cur["range"] = [float(t) for t in v.strip("[]").split()]
        elif k.startswith("MF"):
            name, kind, p = re.match(r"'(.*?)':'(\w+)',\[(.*)\]", v).groups()
            cur["mf"].append(dict(name=name, type=kind, params=[float(t) for t in p.split()]))
    return fis

def writefis(fis, path):
    path = path if path.endswith(".fis") else path + ".fis"
    g = lambda v: f"{v:.15g}"                                   # 15 位有效數字，讀回來不失真
    L = ["[System]", f"Name='{fis['name']}'", f"Type='{fis['type']}'", "Version=2.0", f"NumInputs={len(fis['input'])}",
         f"NumOutputs={len(fis['output'])}", f"NumRules={len(fis['rule'])}", f"AndMethod='{fis['andMethod']}'", f"OrMethod='{fis['orMethod']}'",
         f"ImpMethod='{fis['impMethod']}'", f"AggMethod='{fis['aggMethod']}'", f"DefuzzMethod='{fis['defuzzMethod']}'", ""]
    for key, tag in (("input", "Input"), ("output", "Output")):
        for i, var in enumerate(fis[key]):
            L += [f"[{tag}{i + 1}]", f"Name='{var['name']}'", f"Range=[{' '.join(map(g, var['range']))}]", f"NumMFs={len(var['mf'])}"]
            L += [f"MF{j + 1}='{m['name']}':'{m['type']}',[{' '.join(map(g, m['params']))}]" for j, m in enumerate(var["mf"])] + [""]
    L.append("[Rules]")
    L += [f"{' '.join(map(str, r['antecedent']))}, {' '.join(map(str, r['consequent']))} ({g(r['weight'])}) : {r['connection']}" for r in fis["rule"]]
    open(path, "w", encoding="latin-1").write("\n".join(L) + "\n")
    return path

### 推論：evalfis

* **Mamdani**：各輸入模糊化 → 每條規則的激發強度（AND 取 min 或乘積、OR 取 max 或 probor）× 權重 → 用激發強度截頂（min）或縮放（prod）輸出的歸屬函數 → 所有規則聚合（max、sum 或 probor）→ 解模糊化（centroid、bisector、mom、som、lom）。
* **Sugeno**：每條規則的輸出是常數或輸入的線性函數 z，結果是 Σ wᵢ zᵢ / Σ wᵢ（wtaver）或 Σ wᵢ zᵢ（wtsum）。

輸入超出範圍時以範圍的邊界計算；沒有任何規則被觸發時，輸出取輸出範圍的中點（MATLAB 也是這樣，並顯示警告）。輸入可以是一列（一組輸入）或矩陣（每列一組）。

In [5]:
def _probor(a, b):
    return a + b - a * b

def firing(fis, x):
    ws = []
    for r in fis["rule"]:
        degs = []
        for i, a in enumerate(r["antecedent"]):
            if a == 0:
                continue
            m = fis["input"][i]["mf"][abs(a) - 1]
            d = float(mf(m["type"], m["params"], x[i]))
            degs.append(1 - d if a < 0 else d)
        if r["connection"] == 1:
            w = min(degs) if fis["andMethod"] == "min" else float(np.prod(degs))
        else:
            w = max(degs) if fis["orMethod"] == "max" else float(np.prod([1 - d for d in degs]) * -1 + 1)
        ws.append(r["weight"] * w)
    return np.array(ws)

def defuzz(t, agg, method):
    if agg.sum() == 0:
        return None
    if method == "centroid":
        return float(np.sum(agg * t) / np.sum(agg))
    if method == "bisector":
        k = int(np.argmax(np.cumsum(agg) >= agg.sum() / 2))
        return float(t[k])
    top = t[np.isclose(agg, agg.max())]
    if method == "mom":
        return float(top.mean())
    if method == "som":
        return float(top[np.argmin(np.abs(top))])
    if method == "lom":
        return float(top[np.argmax(np.abs(top))])
    raise ValueError(method)

def evalfis_one(x, fis, npts=101, detail=False):
    x = [min(max(v, var["range"][0]), var["range"][1]) for v, var in zip(x, fis["input"])]
    ws = firing(fis, x)
    out = fis["output"][0]
    lo, hi = out["range"]
    if fis["type"].lower() == "sugeno":
        zs = []
        for r in fis["rule"]:
            m = out["mf"][abs(r["consequent"][0]) - 1]
            zs.append(m["params"][0] if m["type"] == "constant" else float(np.dot(m["params"][:-1], x) + m["params"][-1]))
        zs = np.array(zs)
        if ws.sum() == 0:
            y = (lo + hi) / 2
        else:
            y = float(ws @ zs / ws.sum()) if fis["defuzzMethod"] == "wtaver" else float(ws @ zs)
        return (y, ws, zs) if detail else y
    t = np.linspace(lo, hi, npts)
    clipped = []
    for w, r in zip(ws, fis["rule"]):
        c = r["consequent"][0]
        if c == 0:
            clipped.append(np.zeros(npts)); continue
        m = out["mf"][abs(c) - 1]
        y = mf(m["type"], m["params"], t)
        y = 1 - y if c < 0 else y
        clipped.append(np.minimum(w, y) if fis["impMethod"] == "min" else w * y)
    clipped = np.array(clipped) if clipped else np.zeros((1, npts))
    if fis["aggMethod"] == "max":
        agg = clipped.max(0)
    elif fis["aggMethod"] == "sum":
        agg = clipped.sum(0)
    else:
        agg = np.zeros(npts)
        for c in clipped:
            agg = _probor(agg, c)
    y = defuzz(t, agg, fis["defuzzMethod"])
    if y is None:
        y = (lo + hi) / 2
    return (y, ws, t, clipped, agg) if detail else y

def evalfis(x, fis):
    X = np.atleast_2d(np.asarray(x, float))
    y = np.array([evalfis_one(row, fis) for row in X])
    return float(y[0]) if np.asarray(x).ndim == 1 else y.reshape(-1, 1)

### 檢視：getfis、showrule、plotmf、gensurf、plotfis

In [6]:
def getfis(fis):
    print(f"      Name = {fis['name']}\n      Type = {fis['type']}")
    for key, tag in (("input", "Inputs"), ("output", "Outputs")):
        print(f"      NumInputs = {len(fis['input'])}" if key == "input" else f"      NumOutputs = {len(fis['output'])}")
        for i, v in enumerate(fis[key]):
            print(f"        {tag[:-1]}{i + 1} = {v['name']}  range {v['range']}  MFs: {', '.join(m['name'] for m in v['mf'])}")
    print(f"      NumRules = {len(fis['rule'])}")
    for k in ("andMethod", "orMethod", "impMethod", "aggMethod", "defuzzMethod"):
        print(f"      {k[0].upper() + k[1:]} = {fis[k]}")

def showrule(fis):
    for i, r in enumerate(fis["rule"]):
        conds = []
        for j, a in enumerate(r["antecedent"]):
            if a:
                conds.append(f"({fis['input'][j]['name']} is {'not ' if a < 0 else ''}{fis['input'][j]['mf'][abs(a) - 1]['name']})")
        joint = " and " if r["connection"] == 1 else " or "
        cons = [f"({fis['output'][j]['name']} is {fis['output'][j]['mf'][abs(c) - 1]['name']})" for j, c in enumerate(r["consequent"]) if c]
        print(f"{i + 1}. If {joint.join(conds)} then {' '.join(cons)} ({r['weight']:g})")

def plotmf(fis, var_type, var_index, npts=181):
    var = fis[var_type][var_index - 1]
    x = np.linspace(*var["range"], npts)
    fig = go.Figure()
    for m in var["mf"]:
        if m["type"] in ("constant", "linear"):
            continue
        fig.add_scatter(x=x, y=mf(m["type"], m["params"], x), mode="lines", name=m["name"])
    fig.update_layout(title=f"{var_type} {var_index}: {var['name']}", xaxis_title=var["name"], yaxis_title="Degree of membership",
                      yaxis_range=[-0.02, 1.05], height=300, width=560)
    return fig

def gensurf(fis, n=31):
    v1, v2 = fis["input"][0], fis["input"][1]
    x, y = np.linspace(*v1["range"], n), np.linspace(*v2["range"], n)
    Z = np.array([[evalfis_one([a, b], fis) for a in x] for b in y])
    fig = go.Figure(go.Surface(x=x, y=y, z=np.round(Z, 4), colorscale="Viridis", colorbar_title=fis["output"][0]["name"]))
    fig.update_layout(title=f"gensurf: {fis['name']}", scene=dict(xaxis_title=v1["name"], yaxis_title=v2["name"], zaxis_title=fis["output"][0]["name"]),
                      width=600, height=520)
    return fig

def plotfis(fis):
    """畫出輸入 → FIS → 輸出的方塊圖"""
    fig = go.Figure()
    ni, no = len(fis["input"]), len(fis["output"])
    fig.add_shape(type="rect", x0=0.38, x1=0.62, y0=0.3, y1=0.7, line=dict(width=2), fillcolor="lightyellow")
    fig.add_annotation(x=0.5, y=0.5, text=f"<b>{fis['name']}</b><br>({fis['type']})<br>{len(fis['rule'])} rules", showarrow=False)
    for side, n, x0, key in (("in", ni, 0.05, "input"), ("out", no, 0.75, "output")):
        for i, v in enumerate(fis[key]):
            yc = 0.5 + (n - 1) / 2 * 0.3 - i * 0.3
            fig.add_shape(type="rect", x0=x0, x1=x0 + 0.2, y0=yc - 0.08, y1=yc + 0.08, line=dict(width=1.5), fillcolor="lightblue")
            fig.add_annotation(x=x0 + 0.1, y=yc, text=f"{v['name']} ({len(v['mf'])})", showarrow=False)
            fig.add_shape(type="line", x0=0.25 if side == "in" else 0.62, x1=0.38 if side == "in" else 0.75, y0=yc, y1=0.5, line=dict(width=1.5))
    fig.update_layout(xaxis=dict(visible=False, range=[0, 1]), yaxis=dict(visible=False, range=[0, 1]), width=620, height=300,
                      title=f"System {fis['name']}: {ni} inputs, {no} outputs, {len(fis['rule'])} rules", plot_bgcolor="white")
    return fig

## 2. Fuzzy Logic Toolbox 頁：小費問題

```matlab
fis = readfis('tipper');          % 讀入 tipper.fis
out = evalfis([3 8], fis)         % 計算服務 3、食物 8 時的小費（18.0）
plotmf(fis, 'input', 1)           % 畫第 1 個輸入的歸屬函數
showrule(fis)                     % 以文字列出規則
gensurf(fis)                      % 畫輸出曲面
writefis(fis, 'mytipper')         % 存成 mytipper.fis
```

In [7]:
fis = readfis("tipper")
out = evalfis([3, 8], fis)
print("out =", round(out, 4))
plotmf(fis, "input", 1).show()

out = 18.0059


In [8]:
showrule(fis)
gensurf(fis).show()
writefis(fis, "mytipper")
print(open("mytipper.fis", encoding="latin-1").read())

1. If (service is poor) or (food is rancid) then (tip is cheap) (1)
2. If (service is good) then (tip is average) (1)
3. If (service is excellent) or (food is delicious) then (tip is generous) (1)


[System]
Name='tipper'
Type='mamdani'
Version=2.0
NumInputs=2
NumOutputs=1
NumRules=3
AndMethod='min'
OrMethod='max'
ImpMethod='min'
AggMethod='max'
DefuzzMethod='centroid'

[Input1]
Name='service'
Range=[0 10]
NumMFs=3
MF1='poor':'gaussmf',[1.5 0]
MF2='good':'gaussmf',[1.5 5]
MF3='excellent':'gaussmf',[1.5 10]

[Input2]
Name='food'
Range=[0 10]
NumMFs=2
MF1='rancid':'trapmf',[0 0 1 3]
MF2='delicious':'trapmf',[7 9 10 10]

[Output1]
Name='tip'
Range=[0 30]
NumMFs=3
MF1='cheap':'trimf',[0 5 10]
MF2='average':'trimf',[10 15 20]
MF3='generous':'trimf',[20 25 30]

[Rules]
1 1, 1 (1) : 2
2 0, 2 (1) : 1
3 2, 3 (1) : 2



## 3. 以指令操作 FIS（FisCmdFlash）

```matlab
a = readfis('tipper')         % 讀入 FIS，存成結構變數 a
getfis(a)                     % 列出系統的屬性（型式、變數、規則數……）
showrule(a)                   % 以文字列出規則
plotfis(a)                    % 畫出系統的輸入、輸出方塊圖
plotmf(a, 'input', 1)         % 畫第 1 個輸入（service）的歸屬函數
evalfis([1 2], a)             % 服務 1、食物 2 → 5.5586
evalfis([3 5; 2 7], a)        % 一次算兩組輸入 → [12.2184; 7.7885]
gensurf(a)                    % 畫輸出曲面
a.name = 'mytipper';          % 修改屬性（舊版也可用 setfis）
writefis(a, 'mytipper')       % 存成 mytipper.fis
```

In [9]:
a = readfis("tipper")
getfis(a)
plotfis(a).show()
print("evalfis([1 2], a) =", round(evalfis([1, 2], a), 4))
print("evalfis([3 5; 2 7], a) =\n", np.round(evalfis([[3, 5], [2, 7]], a), 4))
a["name"] = "mytipper"
writefis(a, "mytipper")
print("存檔後的名稱：", readfis("mytipper")["name"])

      Name = tipper
      Type = mamdani
      NumInputs = 2
        Input1 = service  range [0.0, 10.0]  MFs: poor, good, excellent
        Input2 = food  range [0.0, 10.0]  MFs: rancid, delicious
      NumOutputs = 1
        Output1 = tip  range [0.0, 30.0]  MFs: cheap, average, generous
      NumRules = 3
      AndMethod = min
      OrMethod = max
      ImpMethod = min
      AggMethod = max
      DefuzzMethod = centroid


evalfis([1 2], a) = 5.5586
evalfis([3 5; 2 7], a) =
 [[12.2184]
 [ 7.7885]]
存檔後的名稱： mytipper


## 4. 從頭建立 FIS：CreateFIS.m

```matlab
a=newfis('mytipper');
a=addvar(a,'input','service',[0 10]);
a=addmf(a,'input',1,'poor','gaussmf',[1.5 0]);
a=addmf(a,'input',1,'good','gaussmf',[1.5 5]);
a=addmf(a,'input',1,'excellent','gaussmf',[1.5 10]);

a=addvar(a,'input','food',[0 10]);
a=addmf(a,'input',2,'rancid','trapmf',[-2 0 1 3]);
a=addmf(a,'input',2,'delicious','trapmf',[7 9 10 12]);

a=addvar(a,'output','tip',[0 30]);
a=addmf(a,'output',1,'cheep','trimf',[0 5 10]);
a=addmf(a,'output',1,'average','trimf',[10 15 20]);
a=addmf(a,'output',1,'generous','trimf',[20 25 30]);

rulelist=[...
        1 1 1 1 2
        2 0 2 1 1
        3 2 3 1 2];
a=addrule(a,rulelist);

b=evalfis([1 2],a);
c=evalfis([3 5;2 7],a);

writefis(a,'mytipper');
```
（原檔的輸出歸屬函數名稱寫作 `cheep`，照錄。）

In [10]:
a = newfis("mytipper")
a = addvar(a, "input", "service", [0, 10])
a = addmf(a, "input", 1, "poor", "gaussmf", [1.5, 0])
a = addmf(a, "input", 1, "good", "gaussmf", [1.5, 5])
a = addmf(a, "input", 1, "excellent", "gaussmf", [1.5, 10])

a = addvar(a, "input", "food", [0, 10])
a = addmf(a, "input", 2, "rancid", "trapmf", [-2, 0, 1, 3])
a = addmf(a, "input", 2, "delicious", "trapmf", [7, 9, 10, 12])

a = addvar(a, "output", "tip", [0, 30])
a = addmf(a, "output", 1, "cheep", "trimf", [0, 5, 10])
a = addmf(a, "output", 1, "average", "trimf", [10, 15, 20])
a = addmf(a, "output", 1, "generous", "trimf", [20, 25, 30])

rulelist = [[1, 1, 1, 1, 2],
            [2, 0, 2, 1, 1],
            [3, 2, 3, 1, 2]]
a = addrule(a, rulelist)

b = evalfis([1, 2], a)
c = evalfis([[3, 5], [2, 7]], a)
print("b =", round(b, 4))
print("c =\n", np.round(c, 4))
writefis(a, "mytipper")
showrule(a)

b = 5.5586
c =
 [[12.2184]
 [ 7.7885]]
1. If (service is poor) or (food is rancid) then (tip is cheep) (1)
2. If (service is good) then (tip is average) (1)
3. If (service is excellent) or (food is delicious) then (tip is generous) (1)


`rancid` 的梯形延伸到範圍外（[−2 0 1 3]），和 tipper.fis 的 [0 0 1 3] 在 0～10 之間完全一樣，所以結果和 tipper.fis 相同。

## 5. 刪除歸屬函數、變數與規則（FISRMFlash）

```matlab
a = readfis('MyTipper');
a = rmmf(a, 'input', 2, 'mf', 2);     % 刪除第 2 個輸入（food）的第 2 個歸屬函數
a = rmvar(a, 'input', 2);             % 刪除第 2 個輸入變數
a.rule(3) = [];                       % 刪除第 3 條規則
```
規則是以「第幾個歸屬函數」記錄的，刪掉中間的項目會讓編號改變。這裡的 `rmmf` 會把用到被刪除歸屬函數的條件改成 0、比它大的編號減 1，並列出受影響的規則；
MATLAB 不同版本的處理方式不一樣（新版的對應函數是 `removeMF`、`removeInput`、`removeRule`），刪除前最好先檢查規則。

In [11]:
a = readfis("MyTipper")
showrule(a)
print("--- rmmf(a, 'input', 2, 'mf', 2)")
a = rmmf(a, "input", 2, 2)
showrule(a)
print("--- rmvar(a, 'input', 2)")
a = rmvar(a, "input", 2)
showrule(a)
print("--- a.rule(3) = []")
a["rule"].pop(2)
showrule(a)
getfis(a)

1. If (service is poor) or (food is rancid) then (tip is cheep) (1)
2. If (service is good) then (tip is average) (1)
3. If (service is excellent) or (food is delicious) then (tip is generous) (1)
--- rmmf(a, 'input', 2, 'mf', 2)
  注意：規則 3 用到被刪除的歸屬函數，這個條件已改成 0（不用）
1. If (service is poor) or (food is rancid) then (tip is cheep) (1)
2. If (service is good) then (tip is average) (1)
3. If (service is excellent) then (tip is generous) (1)
--- rmvar(a, 'input', 2)
1. If (service is poor) then (tip is cheep) (1)
2. If (service is good) then (tip is average) (1)
3. If (service is excellent) then (tip is generous) (1)
--- a.rule(3) = []
1. If (service is poor) then (tip is cheep) (1)
2. If (service is good) then (tip is average) (1)
      Name = mytipper
      Type = mamdani
      NumInputs = 1
        Input1 = service  range [0.0, 10.0]  MFs: poor, good, excellent
      NumOutputs = 1
        Output1 = tip  range [0.0, 30.0]  MFs: cheep, average, generous
      NumRules = 2
      AndMethod = 

## 6. Mamdani 推論一步一步來（服務 3、食物 8）

[Mamdani 型推論](https://yoshiagent.github.io/IntControl/FZ-mamdani.html)一頁的例子：服務 3 屬於「差」0.135、「好」0.411；食物 8 屬於「好吃」0.5。三條規則的激發強度截頂各自的輸出歸屬函數，聚合後取重心，得到 18.0%。

In [12]:
fis = readfis("tipper")
y, ws, t, clipped, agg = evalfis_one([3, 8], fis, detail=True)
print("激發強度 w =", np.round(ws, 3), "，小費 =", round(y, 4))
fig = make_subplots(rows=1, cols=2, subplot_titles=("each rule after implication (min)", "aggregated (max) and centroid"))
for i, c in enumerate(clipped):
    fig.add_scatter(x=t, y=c, mode="lines", name=f"rule {i + 1} (w={ws[i]:.3f})", row=1, col=1)
fig.add_scatter(x=t, y=agg, mode="lines", fill="tozeroy", name="aggregated", row=1, col=2)
fig.add_vline(x=y, line_dash="dash", annotation_text=f"centroid {y:.2f}", row=1, col=2)
fig.update_xaxes(title_text="tip (%)"); fig.update_layout(height=340)
fig.show()

激發強度 w = [0.135 0.411 0.5  ] ，小費 = 18.0059


### 補充：五種解模糊化方法

同一個聚合結果，用不同的解模糊化方法得到不同的輸出（`DefuzzMethod` 改成 `bisector`、`mom`、`som`、`lom`）：

In [13]:
for method in ("centroid", "bisector", "mom", "som", "lom"):
    f2 = readfis("tipper"); f2["defuzzMethod"] = method
    print(f"{method:9s}：{evalfis([3, 8], f2):.4f}")

centroid ：18.0059
bisector ：18.0000
mom      ：24.9000
som      ：22.5000
lom      ：27.3000


## 7. Sugeno 型的小費問題（tippersg.fis）

輸出是常數（寫成係數為 0 的線性函數：cheap = 5、average = 15、generous = 25），解模糊化是加權平均 `wtaver`。服務 3、食物 8 時為 18.48%：

In [14]:
sg = readfis("tippersg")
getfis(sg)
showrule(sg)
y, ws, zs = evalfis_one([3, 8], sg, detail=True)
print("激發強度 w =", np.round(ws, 3), "，各規則輸出 z =", zs, "，Σwz/Σw =", round(y, 4))
gensurf(sg).show()

      Name = sugeno tipper
      Type = sugeno
      NumInputs = 2
        Input1 = service  range [0.0, 10.0]  MFs: poor, average, good
        Input2 = food  range [0.0, 10.0]  MFs: rancid, delicious
      NumOutputs = 1
        Output1 = tip  range [-30.0, 30.0]  MFs: cheap, average, generous
      NumRules = 3
      AndMethod = min
      OrMethod = max
      ImpMethod = min
      AggMethod = max
      DefuzzMethod = wtaver
1. If (service is poor) or (food is rancid) then (tip is cheap) (1)
2. If (service is average) then (tip is average) (1)
3. If (service is good) or (food is delicious) then (tip is generous) (1)
激發強度 w = [0.135 0.411 0.5  ] ，各規則輸出 z = [ 5. 15. 25.] ，Σwz/Σw = 18.4848


## 8. 沒有規則的練習檔（tipper0.fis）

tipper0.fis 只有變數與歸屬函數、沒有規則，是給學生在 Rule Editor 練習加規則用的。沒有規則時任何輸入都沒有規則觸發，輸出是範圍的中點：

In [15]:
t0 = readfis("tipper0")
print("規則數：", len(t0["rule"]), "，evalfis([3 8]) =", evalfis([3, 8], t0))
t0 = addrule(t0, [[1, 1, 1, 1, 2], [2, 0, 2, 1, 1], [3, 2, 3, 1, 2]])   # 照 tipper.fis 補上三條規則
print("補上規則後 evalfis([3 8]) =", round(evalfis([3, 8], t0), 4))

規則數： 0 ，evalfis([3 8]) = 15.0
補上規則後 evalfis([3 8]) = 18.0059


模糊控制器放進閉迴路的例子（MCK 系統、水槽液位、台車倒單擺）見另一本筆記本 **Fuzzy_control.ipynb**。